# Cycles et demi-cycles du global workspace

In [ ]:
import torch
import torch.nn.functional as F

from numb_project.constants import BASE, GW_CHECKPOINT_FOLDER
from numb_project.notebook_utils import *

torch.set_grad_enabled(False)  # notebook d'évaluation uniquement

gw = load_global_workspace(f"{GW_CHECKPOINT_FOLDER}/rotation.ckpt")

image_mod = gw.gw_mod.domain_mods["image"]

images, labels = mnist_batch(512)
angles = random_angles(len(images))
rotated = rotate_images(images, angles)
datasets_to_test = [("droites", images, torch.zeros_like(angles)), ("tournées", rotated, angles)]

In [ ]:
# --- Cycle image -> digit -> image ---
# Le digit ne garde que le chiffre et l'angle : le style d'écriture est perdu,
# l'image revenue est un chiffre "type" au bon angle. Le digit intermédiaire
# est l'équivalent de la traduction image -> digit.
for name, x, x_angles in datasets_to_test:
    latent = image_mod.encode(x)
    digits_mid, latent_back = cycle(gw, "image", "digit", latent)
    loss = image_mod.compute_loss(latent_back, latent, x).loss
    mse = F.mse_loss(image_mod.decode(latent_back), x)
    print(f"images {name:<8}: loss (latent) = {loss:.4f}   MSE pixels = {mse:.5f}   digit intermédiaire : "
          + format_recognition(digits_mid, labels, x_angles))

latent = image_mod.encode(rotated)
show_rows([
    ("original", rotated),
    ("demi-cycle", image_mod.decode(demi_cycle(gw, "image", latent))),
    ("cycle", image_mod.decode(cycle(gw, "image", "digit", latent)[1])),
])